In [1]:
import copy
import time
import heapq
from collections import deque

# ---------- Constants ----------
EMPTY = " "
PLAYER = "X"
AI = "O"

# ---------- Board Functions ----------
def create_board():
    return [[EMPTY for _ in range(3)] for _ in range(3)]

def print_board(board):
    print("\nBoard:")
    for row in board:
        print(" | ".join(row))
        print("-" * 9)

def get_moves(board):
    moves = []
    for i in range(3):
        for j in range(3):
            if board[i][j] == EMPTY:
                moves.append((i, j))
    return moves

# ---------- Winner Check ----------
def check_winner(board):

    # Rows and Columns
    for i in range(3):
        if board[i][0] == board[i][1] == board[i][2] != EMPTY:
            return board[i][0]
        if board[0][i] == board[1][i] == board[2][i] != EMPTY:
            return board[0][i]

    # Diagonals
    if board[0][0] == board[1][1] == board[2][2] != EMPTY:
        return board[0][0]

    if board[0][2] == board[1][1] == board[2][0] != EMPTY:
        return board[0][2]

    return None

def is_full(board):
    return len(get_moves(board)) == 0

# =====================================================
# BFS SEARCH
# =====================================================

def bfs_search(board):

    start_time = time.time()
    queue = deque([(copy.deepcopy(board), [])])
    visited = set()
    states = 0

    while queue:
        current, path = queue.popleft()
        states += 1

        if check_winner(current) == AI:
            return path, states, time.time() - start_time

        board_tuple = tuple(tuple(r) for r in current)
        if board_tuple in visited:
            continue
        visited.add(board_tuple)

        for move in get_moves(current):
            new_board = copy.deepcopy(current)
            new_board[move[0]][move[1]] = AI
            queue.append((new_board, path + [move]))

    return None, states, time.time() - start_time

# =====================================================
# DFS SEARCH
# =====================================================

def dfs_search(board):

    start_time = time.time()
    stack = [(copy.deepcopy(board), [])]
    visited = set()
    states = 0

    while stack:
        current, path = stack.pop()
        states += 1

        if check_winner(current) == AI:
            return path, states, time.time() - start_time

        board_tuple = tuple(tuple(r) for r in current)
        if board_tuple in visited:
            continue
        visited.add(board_tuple)

        for move in get_moves(current):
            new_board = copy.deepcopy(current)
            new_board[move[0]][move[1]] = AI
            stack.append((new_board, path + [move]))

    return None, states, time.time() - start_time

# =====================================================
# A* SEARCH
# =====================================================

def heuristic(board):
    score = 0
    lines = []

    # rows
    lines.extend(board)

    # columns
    lines.extend([[board[r][c] for r in range(3)] for c in range(3)])

    # diagonals
    lines.append([board[i][i] for i in range(3)])
    lines.append([board[i][2-i] for i in range(3)])

    for line in lines:
        if PLAYER not in line:
            score += 1

    return -score  # negative for priority queue

def astar_search(board):

    start_time = time.time()
    pq = []
    visited = set()
    states = 0

    heapq.heappush(pq, (0, copy.deepcopy(board), []))

    while pq:
        cost, current, path = heapq.heappop(pq)
        states += 1

        if check_winner(current) == AI:
            return path, states, time.time() - start_time

        board_tuple = tuple(tuple(r) for r in current)
        if board_tuple in visited:
            continue
        visited.add(board_tuple)

        for move in get_moves(current):
            new_board = copy.deepcopy(current)
            new_board[move[0]][move[1]] = AI
            h = heuristic(new_board)
            heapq.heappush(pq, (h, new_board, path + [move]))

    return None, states, time.time() - start_time

# =====================================================
# GAME LOOP
# =====================================================

def play_game():

    board = create_board()

    print("Choose AI Algorithm:")
    print("1. BFS")
    print("2. DFS")
    print("3. A*")

    choice = int(input("Enter choice: "))

    while True:

        print_board(board)

        # Player Move
        r = int(input("Enter row (0-2): "))
        c = int(input("Enter col (0-2): "))

        if board[r][c] != EMPTY:
            print("Invalid Move!")
            continue

        board[r][c] = PLAYER

        if check_winner(board) == PLAYER:
            print_board(board)
            print("Player Wins!")
            break

        if is_full(board):
            print("Draw!")
            break

        # AI Move
        if choice == 1:
            path, states, t = bfs_search(board)
        elif choice == 2:
            path, states, t = dfs_search(board)
        else:
            path, states, t = astar_search(board)

        if path:
            r, c = path[0]
            board[r][c] = AI

        print(f"\nAI explored {states} states in {t:.5f} seconds")

        if check_winner(board) == AI:
            print_board(board)
            print("AI Wins!")
            break

        if is_full(board):
            print("Draw!")
            break

# =====================================================
# MAIN
# =====================================================

if __name__ == "__main__":
    play_game()


Choose AI Algorithm:
1. BFS
2. DFS
3. A*

Board:
  |   |  
---------
  |   |  
---------
  |   |  
---------

AI explored 4 states in 0.00024 seconds

Board:
  | X |  
---------
  |   |  
---------
  |   | O
---------


IndexError: list index out of range